<a href="https://colab.research.google.com/github/CH3MLON/Data_GenAI_training/blob/main/day5/day5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")


In [2]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler #labelencoder: converts text categories to integer; standardscaler: scales numbers so all features have same range
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [3]:
ds=pd.read_csv("/content/student_performance.csv")
ds.shape



(30, 13)

In [4]:

ds.columns.tolist()


['student_id',
 'name',
 'age',
 'gender',
 'department',
 'semester',
 'math_score',
 'science_score',
 'english_score',
 'programming_score',
 'attendance_percentage',
 'city',
 'admission_year']

In [5]:

ds.isnull().sum().sum()

np.int64(0)

In [6]:
ds.duplicated(subset=["name","admission_year"])#so no duplicates


,0
0,False
1,False
2,False
3,False
4,False
5,False
6,False
7,False
8,False
9,False


In [7]:
ds['programming_score'].describe()

,programming_score
count,30.000000
mean,67.600000
std,21.041175
min,38.000000
25%,49.250000
50%,66.000000
75%,88.750000
max,97.000000


In [8]:
ds_ml=ds.copy()
le_gender=LabelEncoder()

In [9]:
ds_ml['gender_enc']=le_gender.fit_transform(ds_ml['gender'])
print(f"gender encoded: {dict(zip(le_gender.classes_,le_gender.transform(le_gender.classes_)))}")


gender encoded: {'Female': np.int64(0), 'Male': np.int64(1)}


In [10]:
le_dept=LabelEncoder()
ds_ml['dept_enc']=le_dept.fit_transform(ds_ml['department'])
print(f"dept encoded: {dict(zip(le_dept.classes_,le_dept.transform(le_dept.classes_)))}")

dept encoded: {'Civil': np.int64(0), 'Computer Science': np.int64(1), 'Electronics': np.int64(2), 'Mechanical': np.int64(3)}


In [11]:
print("\n New columns added")
ds_ml[['gender','gender_enc','department','dept_enc']].head()


 New columns added


,gender,gender_enc,department,dept_enc
0,Male,1,Computer Science,1
1,Female,0,Computer Science,1
2,Male,1,Electronics,2
3,Female,0,Mechanical,3
4,Male,1,Computer Science,1


In [12]:
ds_ml.columns.tolist()

['student_id',
 'name',
 'age',
 'gender',
 'department',
 'semester',
 'math_score',
 'science_score',
 'english_score',
 'programming_score',
 'attendance_percentage',
 'city',
 'admission_year',
 'gender_enc',
 'dept_enc']

In [13]:
f_cols=[
 'math_score',
 'science_score',
 'english_score',
 'attendance_percentage',
 'gender_enc',
 'dept_enc'
]
X=ds_ml[f_cols]
y=ds_ml['programming_score']
print(f"feature matrix X:{X.shape} ")
print(f"target vector y:{y.shape}")
print(f"feature:{f_cols}")


feature matrix X:(30, 6) 
target vector y:(30,)
feature:['math_score', 'science_score', 'english_score', 'attendance_percentage', 'gender_enc', 'dept_enc']


In [14]:
y.describe()

,programming_score
count,30.000000
mean,67.600000
std,21.041175
min,38.000000
25%,49.250000
50%,66.000000
75%,88.750000
max,97.000000


In [15]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)
print(f"X_train:{X_train.shape}")
print(f"X_test:{X_test.shape}")
print(f"y_train:{y_train.shape}")
print(f"y_test:{y_test.shape}")
print(f"X_train:{len(X_train)}({len(X_train)/len(X)*100}%)")
print(f"X_test:{len(X_test)}({len(X_test)/len(X)*100}%)")
print(f"y_train:{len(y_train)}({len(y_train)/len(y)*100}%)")
print(f"y_test:{len(y_test)}({len(y_test)/len(y)*100}%)")

X_train:(24, 6)
X_test:(6, 6)
y_train:(24,)
y_test:(6,)
X_train:24(80.0%)
X_test:6(20.0%)
y_train:24(80.0%)
y_test:6(20.0%)


In [16]:
scaler=StandardScaler()
X_train_scale=scaler.fit_transform(X_train)
X_test_scale=scaler.transform(X_test)







### Train and Evaluate Linear Regression Model

In [17]:
lr_model = LinearRegression()
lr_model.fit(X_train_scale, y_train)

LinearRegression()

In [18]:
y_pred_lr = lr_model.predict(X_test_scale)

mae_lr = mean_absolute_error(y_test, y_pred_lr)
mse_lr = mean_squared_error(y_test, y_pred_lr)
r2_lr = r2_score(y_test, y_pred_lr)

print(f"Linear Regression - MAE: {mae_lr:.2f}")
print(f"Linear Regression - MSE: {mse_lr:.2f}")
print(f"Linear Regression - R2 Score: {r2_lr:.2f}")

Linear Regression - MAE: 9.37
Linear Regression - MSE: 131.54
Linear Regression - R2 Score: 0.74


### Predict Programming Score for Custom Input

In [19]:

custom_input = {
    'math_score': 80,
    'science_score': 75,
    'english_score': 70,
    'attendance_percentage': 85,
    'gender_enc': 1,  # 0 for Female, 1 for Male
    'dept_enc': 1     # 0 for Civil, 1 for Computer Science, 2 for Electronics, 3 for Mechanical
}

# Convert to DataFrame
custom_df = pd.DataFrame([custom_input])

# Scale the custom input using the same scaler fitted on training data
custom_scaled = scaler.transform(custom_df)

# Predict the programming score
predicted_score = lr_model.predict(custom_scaled)

print(f"Predicted Programming Score: {predicted_score[0]:.2f}")

Predicted Programming Score: 72.41


### Understanding Model 'Accuracy' for Regression

When evaluating regression models, the term 'accuracy' can be misleading as it's typically used for classification tasks. For regression, we assess how well the model's predictions align with the actual continuous values. The metrics we use are:

*   **Mean Absolute Error (MAE):** The average absolute difference between predicted and actual values. A lower MAE indicates better predictions.
*   **Mean Squared Error (MSE):** The average of the squared differences between predicted and actual values. It penalizes larger errors more heavily. A lower MSE indicates better predictions.
*   **R-squared (R2 Score):** This is the proportion of the variance in the dependent variable that is predictable from the independent variables. It ranges from 0 to 1. An R2 score closer to 1 indicates that the model explains a larger portion of the variance in the target variable, essentially giving you a measure of how 'accurate' the model is in explaining the data's variability. A score of 0.74 means your model explains 74% of the variance in programming scores.

Based on your previous Linear Regression results:

*   **MAE: 9.37**
*   **MSE: 131.54**
*   **R2 Score: 0.74**

These metrics provide a comprehensive view of your model's performance, with the R2 score being a good indicator of its explanatory power.

### Train and Evaluate Decision Tree Regressor Model

In [20]:
dt_model = DecisionTreeRegressor(random_state=42)
dt_model.fit(X_train_scale, y_train)

DecisionTreeRegressor(random_state=42)

In [21]:
y_pred_dt = dt_model.predict(X_test_scale)

mae_dt = mean_absolute_error(y_test, y_pred_dt)
mse_dt = mean_squared_error(y_test, y_pred_dt)
r2_dt = r2_score(y_test, y_pred_dt)

print(f"Decision Tree Regressor - MAE: {mae_dt:.2f}")
print(f"Decision Tree Regressor - MSE: {mse_dt:.2f}")
print(f"Decision Tree Regressor - R2 Score: {r2_dt:.2f}")

Decision Tree Regressor - MAE: 10.67
Decision Tree Regressor - MSE: 231.67
Decision Tree Regressor - R2 Score: 0.53


### Train and Evaluate Random Forest Regressor Model

In [22]:
rf_model = RandomForestRegressor(random_state=42)
rf_model.fit(X_train_scale, y_train)

RandomForestRegressor(random_state=42)

In [23]:
y_pred_rf = rf_model.predict(X_test_scale)

mae_rf = mean_absolute_error(y_test, y_pred_rf)
mse_rf = mean_squared_error(y_test, y_pred_rf)
r2_rf = r2_score(y_test, y_pred_rf)

print(f"Random Forest Regressor - MAE: {mae_rf:.2f}")
print(f"Random Forest Regressor - MSE: {mse_rf:.2f}")
print(f"Random Forest Regressor - R2 Score: {r2_rf:.2f}")

Random Forest Regressor - MAE: 10.60
Random Forest Regressor - MSE: 195.04
Random Forest Regressor - R2 Score: 0.61
